# Notebook 4: Metric View Generation
**Auto-Genie — Genie Space Creation: Stage 4**

This notebook generates draft Databricks Metric View YAML definitions for each
configured table. It runs independently of the Genie deployment notebook and only
requires table metadata and column statistics.

**Pipeline:**
```
Notebook 01 — Query Intelligence     → example_queries.json
Notebook 02 — Assembly & Validation  → knowledge_store.json
Notebook 03 — Genie Deployment       → Live Genie Space
Notebook 04 — Metric View Generation → metric_views/*.yml   [THIS NOTEBOOK]
```

**Outputs** written to `metric_view_output_path` (config):
- `{table}_metrics.yml` — per-table Metric View YAML
- `all_metric_views.yml` — combined file
- `metric_view_report.json` — validation + classification summary

## Cell 4.0: Environment & Configuration

In [ ]:
import sys, os, json
from pathlib import Path
from datetime import datetime

sys.path.insert(0, str(Path.cwd().parent / "utils"))
from auto_genie_utils import (
    load_yaml_config,
    load_prompts,
    get_spark_session,
    extract_table_metadata,
    profile_column_statistics,
    discover_naming_pattern_relationships,
    classify_columns_for_metric_view,
    detect_table_roles,
    enrich_metric_view_with_llm,
    build_metric_view_dict,
    serialize_and_validate_metric_view,
)

config  = load_yaml_config()
prompts = load_prompts()
spark   = get_spark_session(config)

print(f"✅ Config loaded — targeting {config['catalog']}.{config['schema']}")
print(f"   Metric view mode : {config.get('metric_view_mode', 'per_table')}")
print(f"   Output path      : {config.get('metric_view_output_path', config['output_path'] + '/metric_views')}")

## Cell 4.1: Load Table Metadata & Profiles

In [ ]:
print("=" * 60)
print("LOADING TABLE METADATA & STATISTICS")
print("=" * 60)

table_metadata = extract_table_metadata(
    spark, config['catalog'], config['schema'], config['tables']
)
column_profiles = profile_column_statistics(spark, table_metadata)
relationships   = discover_naming_pattern_relationships(table_metadata)

total_cols = sum(len(m['columns']) for m in table_metadata.values())
print(f"\n✅ Loaded {len(table_metadata)} tables, {total_cols} columns, "
      f"{sum(len(v) for v in column_profiles.values())} column profiles")
print(f"   Relationships discovered: {len(relationships)}")

## Cell 4.2: Rule-Based Column Classification

In [ ]:
print("=" * 60)
print("RULE-BASED COLUMN CLASSIFICATION")
print("=" * 60)

column_classifications = classify_columns_for_metric_view(
    table_metadata, column_profiles, config
)

# ── Summary table ────────────────────────────────────────────────────────────
header = f"{'Table':<40} {'TimeDims':>9} {'Dims':>6} {'Measures':>9} {'IDs':>6} {'Skipped':>8}"
print(f"\n{header}")
print("-" * len(header))
for fqn, cls in column_classifications.items():
    simple = fqn.split(".")[-1]
    print(
        f"{simple:<40}"
        f"{len(cls['time_dimensions']):>9}"
        f"{len(cls['dimensions']):>6}"
        f"{len(cls['measures']):>9}"
        f"{len(cls['identifiers']):>6}"
        f"{len(cls['skipped']):>8}"
    )

# ── Detail view ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("CLASSIFICATION DETAIL")
print("=" * 60)
for fqn, cls in column_classifications.items():
    simple = fqn.split(".")[-1]
    print(f"\n  {simple.upper()}")
    for bucket, cols in cls.items():
        if cols:
            names = ", ".join(c["name"] for c in cols)
            print(f"    {bucket:<18}: {names}")

## Cell 4.3: Table Role Detection

In [ ]:
print("=" * 60)
print("TABLE ROLE DETECTION")
print("=" * 60)

table_roles = detect_table_roles(
    table_metadata, relationships, column_classifications
)

print()
for fqn, role in table_roles.items():
    icon   = "📊" if role == "fact" else "📋"
    simple = fqn.split(".")[-1]
    n_rows = table_metadata[fqn].get("row_count", 0)
    n_meas = len(column_classifications.get(fqn, {}).get("measures", []))
    n_dims = len(column_classifications.get(fqn, {}).get("dimensions", []))
    print(
        f"  {icon}  {simple:<40} → {role.upper():<10} "
        f"({n_rows:>7,} rows | {n_meas} measures | {n_dims} dims)"
    )

fact_tables = [fqn for fqn, r in table_roles.items() if r == "fact"]
dim_tables  = [fqn for fqn, r in table_roles.items() if r == "dimension"]
print(f"\n✅ Fact tables: {len(fact_tables)} | Dimension tables: {len(dim_tables)}")

## Cell 4.4: LLM Semantic Enrichment

Calls the LLM once per table to enrich each classified column with:
- `comment` — 1-sentence business description (written to YAML)
- `aggregation` — confirmed aggregation for measures (SUM / AVG / COUNT / etc.)
- `exclude` — flag columns with no analytical value

Falls back to rule-based enrichment (snake_case → Title Case) if the LLM is
unavailable or blocked by guardrails.

In [ ]:
print("=" * 60)
print("LLM SEMANTIC ENRICHMENT")
print("=" * 60)

all_enrichments: dict = {}

for fqn in table_metadata:
    simple = fqn.split(".")[-1]
    role   = table_roles.get(fqn, "fact")
    print(f"\n🤖 Enriching {simple} ({role})...")

    enrichment = enrich_metric_view_with_llm(
        table_fqn=fqn,
        column_classifications=column_classifications,
        table_metadata=table_metadata,
        prompts_cfg=prompts,
        config=config,
        table_role=role,
    )
    all_enrichments[fqn] = enrichment

    n_cols    = sum(1 for k in enrichment if not k.startswith("__"))
    n_exclude = sum(1 for v in enrichment.values() if isinstance(v, dict) and v.get("exclude"))
    print(f"   ✅ {n_cols} columns enriched, {n_exclude} flagged for exclusion")
    if enrichment.get("__table__"):
        comment = enrichment["__table__"].get("description", "")
        print(f"   📝 View comment: {comment[:100]}{'...' if len(comment) > 100 else ''}")

print(f"\n✅ LLM enrichment complete for {len(all_enrichments)} tables")

## Cell 4.5: Build Metric View Dicts

In [ ]:
print("=" * 60)
print("BUILDING METRIC VIEW DICTS")
print("=" * 60)

metric_view_dicts: dict = {}

for fqn in table_metadata:
    simple = fqn.split(".")[-1]
    role   = table_roles.get(fqn, "fact")

    mv_dict = build_metric_view_dict(
        table_fqn=fqn,
        table_role=role,
        column_classifications=column_classifications,
        llm_enrichments=all_enrichments.get(fqn, {}),
        relationships=relationships,
        config=config,
    )
    metric_view_dicts[fqn] = mv_dict

    n_dims  = len(mv_dict.get("dimensions", []))
    n_meas  = len(mv_dict.get("measures", []))
    n_joins = len(mv_dict.get("joins", []))
    view_name = mv_dict.get("_view_name", simple + "_metrics")
    print(
        f"\n  {view_name}  ({role})"
        f"\n    source     : {mv_dict['source']}"
        f"\n    dimensions : {n_dims}"
        f"\n    measures   : {n_meas}"
        + (f"\n    joins      : {n_joins}" if n_joins else "")
    )

print(f"\n✅ Built {len(metric_view_dicts)} metric view dicts")

## Cell 4.6: Serialize & Validate YAML

Each dimension and measure expression is validated by wrapping it in
`SELECT {expr} FROM {source} LIMIT 1` and parsing it with `sqlglot`.

In [ ]:
print("=" * 60)
print("SERIALIZE & VALIDATE YAML")
print("=" * 60)

serialized: dict = {}

for fqn, mv_dict in metric_view_dicts.items():
    simple = fqn.split(".")[-1]
    result = serialize_and_validate_metric_view(mv_dict)
    serialized[fqn] = result

    status = "✅ PASSED" if result["passed"] else "⚠️  SOME FAILED"
    print(f"\n  {simple}: {status}")

    for bucket, items in result["validation"].items():
        if not items:
            continue
        n_pass = sum(1 for v in items.values() if v == "passed")
        print(f"    {bucket:<18}: {n_pass}/{len(items)} passed")
        for col, outcome in items.items():
            if outcome != "passed":
                print(f"      ❌ {col}: {outcome}")

# ── YAML preview of first table ───────────────────────────────────────────────
first_fqn = next(iter(serialized))
first_name = first_fqn.split(".")[-1]
print(f"\n{'=' * 60}")
print(f"YAML PREVIEW — {first_name}")
print("=" * 60)
print(serialized[first_fqn]["yaml_string"])

# ── Overall summary ───────────────────────────────────────────────────────────
all_passed = all(r["passed"] for r in serialized.values())
print(f"\n{'=' * 60}")
print(f"VALIDATION SUMMARY")
print("=" * 60)
for fqn, result in serialized.items():
    view_name = result["view_name"] or fqn.split(".")[-1] + "_metrics"
    total_items = sum(len(v) for v in result["validation"].values())
    total_pass  = sum(
        sum(1 for x in v.values() if x == "passed")
        for v in result["validation"].values()
    )
    icon = "✅" if result["passed"] else "⚠️ "
    print(f"  {icon} {view_name:<40} {total_pass}/{total_items} expressions valid")
print(f"\nOverall: {'✅ ALL VALID' if all_passed else '⚠️  REVIEW FAILURES ABOVE'}")

## Cell 4.7: Save Outputs

Writes:
- One `.yml` file per table (e.g. `opportunity_metrics.yml`)
- `all_metric_views.yml` — all views concatenated
- `metric_view_report.json` — validation + classification summary

Also prints the `CREATE VIEW ... WITH METRICS` DDL for each table, ready to
paste into a Databricks notebook or SQL editor.

In [ ]:
print("=" * 60)
print("SAVING OUTPUTS")
print("=" * 60)

output_dir = Path(
    config.get("metric_view_output_path", config["output_path"] + "/metric_views")
    .replace("/dbfs", "")
)
output_dir.mkdir(parents=True, exist_ok=True)

saved_files      = []
validation_report: dict = {}

# ── Per-table YAML files ──────────────────────────────────────────────────────
for fqn, result in serialized.items():
    view_name = result["view_name"] or fqn.split(".")[-1] + "_metrics"
    out_file  = output_dir / f"{view_name}.yml"
    out_file.write_text(result["yaml_string"])
    saved_files.append(str(out_file))

    cls = column_classifications.get(fqn, {})
    validation_report[view_name] = {
        "fqn"        : fqn,
        "view_name"  : view_name,
        "table_role" : table_roles.get(fqn, "unknown"),
        "passed"     : result["passed"],
        "n_dimensions": len(metric_view_dicts[fqn].get("dimensions", [])),
        "n_measures"  : len(metric_view_dicts[fqn].get("measures", [])),
        "classification_summary": {
            bucket: len(cols) for bucket, cols in cls.items()
        },
        "validation" : result["validation"],
    }
    print(f"  💾 {out_file.name}")

# ── Combined YAML ─────────────────────────────────────────────────────────────
combined_lines = [
    f"# Auto-Genie — Combined Metric Views",
    f"# Generated : {datetime.now().isoformat()}",
    f"# Catalog   : {config['catalog']}.{config['schema']}",
    "",
]
for fqn, result in serialized.items():
    view_name = result["view_name"] or fqn.split(".")[-1] + "_metrics"
    combined_lines.append(f"# {'─' * 50}")
    combined_lines.append(f"# {view_name}  (source: {fqn})")
    combined_lines.append(f"# {'─' * 50}")
    combined_lines.append(result["yaml_string"])

combined_file = output_dir / "all_metric_views.yml"
combined_file.write_text("\n".join(combined_lines))
saved_files.append(str(combined_file))
print(f"  💾 {combined_file.name}")

# ── Validation report ─────────────────────────────────────────────────────────
report_file = output_dir / "metric_view_report.json"
report_file.write_text(json.dumps(validation_report, indent=2))
saved_files.append(str(report_file))
print(f"  💾 {report_file.name}")

# ── Final summary ─────────────────────────────────────────────────────────────
all_passed = all(r["passed"] for r in validation_report.values())
print(f"\n{'=' * 60}")
print("METRIC VIEW GENERATION COMPLETE")
print("=" * 60)
print(f"  Output directory : {output_dir}")
print(f"  Files saved      : {len(saved_files)}")
print(f"  Overall status   : {'✅ ALL EXPRESSIONS VALID' if all_passed else '⚠️  REVIEW FAILURES IN REPORT'}")

print(f"\n{'=' * 60}")
print("DEPLOYMENT DDL")
print("=" * 60)
print("Run the following in a Databricks notebook (Runtime 17.2+):")
cat    = config['catalog']
schema = config['schema']
for fqn, result in serialized.items():
    view_name  = result["view_name"] or fqn.split(".")[-1] + "_metrics"
    yaml_body  = result["yaml_string"].strip()
    print(f"\n# {view_name}")
    print(f"spark.sql(\"\"\"")
    print(f"  CREATE OR REPLACE VIEW {cat}.{schema}.{view_name}")
    print(f"  WITH METRICS LANGUAGE YAML AS $$")
    for line in yaml_body.splitlines():
        print(f"    {line}")
    print(f"  $$")
    print(f"\"\"\")")  